# COVID-19 Research Literature - Topic Modeling

This notebook applies **Latent Dirichlet Allocation (LDA)** to a corpus of pre-pandemic COVID-19 and coronavirus research papers.
The analysis pipeline follows:
**Dataset Inventory -> Preprocessing (`helper_functions.py`) -> Vectorization -> LDA Model Building (8, 9, and 10 Topics) -> Quantitative Evaluation (Perplexity + NPMI Coherence) -> Topic Interpretation**
> **Preprocessing Note**
>
> The `process_articles` class implemented in `helper_functions.py` performs the following preprocessing steps within the `process_text()` method:
>
> 1. Lowercasing and regex-based noise removal (HTML tags, URLs, numbers, and special characters)
> 2. Tokenization using Gensim `simple_preprocess()`, which also removes punctuation
> 3. Stop word removal using the NLTK English stop word list supplied during object construction
> 4. **Porter Stemming**, reducing each token to its root form (e.g., *infections* → `infect`, *patients* → `patient`)
>
> During vectorization, an additional **multilingual and domain-specific stop word list** is applied within `CountVectorizer` to suppress residual non-informative terms and improve topic interpretability.


## Step 1 - Dataset Inventory
Walk the four corpus subdirectories and count the available JSON article files before loading anything.

In [1]:
import os
# Root folder containing all four corpus subdirectories
corpus_root = r"C:\Users\Sthuti\Desktop\Week5\assignment_5_data"
total_json_files = 0
for directory_path, subdirectories, filenames in os.walk(corpus_root):
    json_files_in_dir = [f for f in filenames if f.endswith(".json")]
    if json_files_in_dir:
        print(f"{directory_path} -> {len(json_files_in_dir)} JSON files")
        total_json_files += len(json_files_in_dir)
print(f"\nTotal JSON files: {total_json_files}")

C:\Users\Sthuti\Desktop\Week5\assignment_5_data\biorxiv_medrxiv\biorxiv_medrxiv -> 885 JSON files
C:\Users\Sthuti\Desktop\Week5\assignment_5_data\comm_use_subset\comm_use_subset -> 9118 JSON files
C:\Users\Sthuti\Desktop\Week5\assignment_5_data\custom_license\custom_license -> 16959 JSON files
C:\Users\Sthuti\Desktop\Week5\assignment_5_data\noncomm_use_subset\noncomm_use_subset -> 2353 JSON files

Total JSON files: 29315


## Step 2 - Imports and Corpus Paths
All four subdirectory paths are declared here so they can be passed to `process_articles`.  
`helper_functions.py` imports are made with `*` as instructed by the assignment to expose `process_articles`, `LDA_Evaluator`, and `word_clouds`.

In [2]:
from helper_functions import *   # exposes process_articles, LDA_Evaluator, wcEval, word_clouds
# Absolute paths to the four article subdirectories
corpus_subdirectory_paths = [
    r"C:\Users\Sthuti\Desktop\Week5\assignment_5_data\biorxiv_medrxiv\biorxiv_medrxiv",
    r"C:\Users\Sthuti\Desktop\Week5\assignment_5_data\comm_use_subset\comm_use_subset",
    r"C:\Users\Sthuti\Desktop\Week5\assignment_5_data\custom_license\custom_license",
    r"C:\Users\Sthuti\Desktop\Week5\assignment_5_data\noncomm_use_subset\noncomm_use_subset"
]

## Step 3 - Preprocessing Pipeline
### What `process_articles` does internally
| Stage | Operation |
|---|---|
| `read_files()` | Walks each subdirectory, samples ~20% of files at random, loads JSON, extracts title + body text |
| `process_text()` | Lowercase -> regex clean -> gensim tokenize -> NLTK stop word removal -> **Porter Stemming** |

The Porter Stemmer reduces inflected words to their root form so the vectorizer treats *"infection"*, *"infected"*, and *"infecting"* as the same token (`infect`).  
This is critical for LDA because it reduces vocabulary sparsity and groups semantically related terms together.

The NLTK English stop word list is supplied at construction time so that common function words are stripped before stemming.

In [3]:
import nltk
from nltk.corpus import stopwords
# Download NLTK stop-word list (no-op if already present)
nltk.download('stopwords')
# English stop words passed into process_articles for the initial cleaning pass
nltk_english_stopwords = stopwords.words('english')
# Instantiate the article processor with corpus paths and stop word list
article_processor = process_articles(
    file_locs=corpus_subdirectory_paths,
    stopwords=nltk_english_stopwords
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sthuti\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
# read raw files from disk
# Samples ~20% of files from each subdirectory -> controlled inside helper_functions
article_processor.read_files()
# apply the full preprocessing pipeline
# Internally: lowercase -> regex clean -> tokenize -> stop word removal -> Porter Stemming
article_processor.process_text()
# Each entry in processed_article is a 4-tuple:
#   [0] subdirectory name (source label)
#   [1] article title
#   [2] raw body text (list of paragraph strings)
#   [3] stemmed token list  <-- used for LDA
total_processed_articles = len(article_processor.processed_article)
print(total_processed_articles)

Processing Files at the requested location
There are 177 files to process
There were 885 files in the dataset
Processing Files at the requested location
There are 1823 files to process
There were 9118 files in the dataset
Processing Files at the requested location
There are 3391 files to process
There were 16959 files in the dataset
Processing Files at the requested location
There are 470 files to process
There were 2353 files in the dataset
Cleaning out Junk
Tokenizing words
Converting to list of words and removing stop words
Creating word stems
5861


## Step 4 - Persist and Reload Processed Articles
Pickling the `process_articles` object avoids re running the expensive preprocessing pipeline on every kernel restart.  
The saved file can be loaded directly in subsequent sessions.

In [5]:
import pickle
PICKLE_PATH = "processed_articles.pkl"
# save the fully preprocessed article_processor object to disk
try:
    with open(PICKLE_PATH, "wb") as pickle_file:
        pickle.dump(article_processor, pickle_file)
    print("Saved successfully.")
except OSError as io_error:
    print(f"Could not save pickle file: {io_error}")

Saved successfully.


In [6]:
import pickle
PICKLE_PATH = "processed_articles.pkl"
#reload the preprocessed object -> run this cell instead of Steps 3-4 on subsequent sessions
try:
    with open(PICKLE_PATH, "rb") as pickle_file:
        article_processor = pickle.load(pickle_file)
    print(f"Loaded {len(article_processor.processed_article)} preprocessed articles.")
except FileNotFoundError:
    print("Pickle file not found!!!.Run the preprocessing cells above first.")
except Exception as load_error:
    print(f"Failed to load pickle: {load_error}")

Loaded 5861 preprocessed articles.


## Step 5 - Inspect a Preprocessed Article
Printing the first entry confirms the 4 tuple structure and lets us visually verify that stemming has been applied correctly before proceeding to vectorization.

In [7]:
# Verify the structure of a single preprocessed article
# Structure: [source_label, title, [raw_paragraphs], [stemmed_tokens]]
# index [3] is the stemmed token list used for LDA
sample_article = article_processor.processed_article[0]
print(sample_article)

['biorxiv_medrxiv', 'VADR: validation and annotation of virus sequence submissions to GenBank', ['As of September 2019, GenBank [1] contains more than 3 million viral sequences totaling over 4 billion nucleotides in length and including over 180,000 complete genomes for viruses other than influenza. More than 250,000 of these sequences were submitted in 2018. All sequence submissions are validated prior to deposition in GenBank. Automated validation and annotation methods become increasingly important as sequence submission numbers grow. Table 1 shows the number of sequences for the 16 virus species with the most sequences in GenBank. Influenza sequences are the second most abundant and the National Center of Biotechnology Information (NCBI), where GenBank is housed, has expended considerable effort to organize flu sequences and streamline the submission of new influenza virus sequences, including a tool to validate and annotate flu submissions called FLAN [2] . The influenza virus seq

## Step 6 - Build Token Strings for CountVectorizer
CountVectorizer expects a list of strings (one per document), so the stemmed token list at index `[3]` of each article tuple is rejoined into a single space separated string.

In [8]:
# rejoin stemmed token list into a single string per document
# index [3] = Porter stemmed tokens after stop word removal
stemmed_document_strings = [
    " ".join(article[3])
    for article in article_processor.processed_article
]

print("Number of documents:", len(stemmed_document_strings))
print("\nSample document (first 500 chars):\n")
print(stemmed_document_strings[0][:500])

Number of documents: 5861

Sample document (first 500 chars):

septemb genbank contain million viral sequenc total billion nucleotid length includ complet genom virus influenza sequenc submit sequenc submiss valid prior deposit genbank autom valid annot method becom increasingli import sequenc submiss number grow tabl show number sequenc viru speci sequenc genbank influenza sequenc second abund nation center biotechnolog inform ncbi genbank hous expend consider effort organ flu sequenc streamlin submiss new influenza viru sequenc includ tool valid annot flu


## Step 7 - Extended Stop-Word List
The corpus contains papers in multiple languages (English, Spanish, French) and many domain generic scientific phrases ("figure", "table", "result") that would add noise to topics without adding meaning.  
A second stop word pass is applied inside `CountVectorizer` to suppress these.
Four categories are merged:
- **English** — NLTK built in
- **Spanish** — function words appearing in multilingual papers
- **French** — same rationale
- **Scientific domain** — generic methodology words that appear across nearly every topic
Note: `"dan"` (Indonesian/Malay for *and*) is included in the scientific list to fix the raw frequency artifact observed in Topic 6.

In [9]:
from nltk.corpus import stopwords
# english stop words -> NLTK
english_stopwords = stopwords.words("english")
# spanish function words observed in multilingual abstracts
spanish_stopwords = [
    "de", "la", "el", "en", "lo", "que", "se", "con", "por", "del",
    "los", "las", "para", "una", "un", "es", "como", "su", "sus",
    "al", "más", "mas", "entre", "sobre", "también", "tambien",
    "sin", "hasta", "desde", "todo", "todos", "esta", "este",
    "estas", "estos", "hay", "son", "fue", "ser", "han"
]
# french function words observed in multilingual abstracts 
french_stopwords = [
    "le", "la", "les", "de", "des", "du", "et", "une", "un",
    "est", "en", "pour", "dans", "sur", "avec", "par",
    "au", "aux", "ce", "ces", "cet", "cette", "ou",
    "où", "plus", "moins", "comme", "sans", "être",
    "etre", "ont", "sont", "avait", "avoir", "fait"
]
# scientific methodology words that are not topic discriminative
# 'dan' is Indonesian/Malay for 'and'  suppressed to fix Topic 6 artifact
scientific_stopwords = [
    "al", "et", "fig", "figure", "table",
    "use", "used", "using",
    "may", "also", "show", "shown",
    "result", "results", "method", "methods",
    "study", "studies",
    "one", "two", "three",
    "new", "data", "analysis",
    "respectively", "however",
    "could", "would", "found",
    "include", "including",
    "among", "within", "well",
    "present", "previous", "based",
    "dan", "pr", "ne", "plu"  
]
# Merge all four lists and deduplicate
combined_stopwords = list(
    set(english_stopwords + spanish_stopwords + french_stopwords + scientific_stopwords)
)
print(f"Total stop words: {len(combined_stopwords)}")

Total stop words: 308


## Step 8 - Vectorization with CountVectorizer
The stemmed document strings are converted into a **document-term matrix** using `CountVectorizer`.
| Parameter | Value | Rationale |
|---|---|---|
| `max_df=0.95` | Ignore terms in >95% of documents | Removes near universal corpus words that provide no discriminative signal |
| `min_df=5` | Ignore terms appearing in fewer than 5 documents | Removes extremely rare words that are likely OCR noise or unique proper nouns |
| `stop_words` | Combined 308 word list | Second stop word pass after stemming removes multilingual and domain generic noise |
The resulting matrix shape `(n_documents, n_vocabulary_terms)` is printed as a sanity check.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
# Build document term matrix from stemmed document strings
# max_df=0.95: drop words in >95% of documents (near universal, no discriminative power)
# min_df=5:    drop words in fewer than 5 documents (rare noise)
corpus_vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=5,
    stop_words=combined_stopwords
)
document_term_matrix = corpus_vectorizer.fit_transform(stemmed_document_strings)
n_docs, n_terms = document_term_matrix.shape
print(f"Document-term matrix shape: {document_term_matrix.shape}")
print(f"  -> {n_docs} documents  x  {n_terms} vocabulary terms")
# Retrieve vocabulary terms aligned with CountVectorizer column indices
vocabulary_feature_names = corpus_vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocabulary_feature_names)} terms")

Document-term matrix shape: (5861, 35779)
  -> 5861 documents  x  35779 vocabulary terms
Vocabulary size: 35779 terms


## Step 9 - Train LDA Models (8, 9, and 10 Topics)
Three LDA models are trained inside a loop and stored in a dictionary keyed by topic count.
**Key hyperparameter choices:**
- `random_state=42` — ensures reproducibility across runs
- `learning_method='batch'` — uses the full dataset on each EM iteration; `'online'` would be preferred for very large corpora that do not fit in memory
- `max_iter=10` (sklearn default) — standard for exploratory topic modeling
**Perplexity** is computed for each model as the first quantitative evaluation metric.  
Lower perplexity = better statistical fit to the held-in data.

In [11]:
from sklearn.decomposition import LatentDirichletAllocation
# Storage for trained models and their perplexity scores
trained_lda_models   = {}   # key: n_topics (int) -> value: fitted LDA model
model_perplexity     = {}   # key: n_topics (int) -> value: perplexity score (float)
for n_topics in [8, 9, 10]:
    print(f"\nTraining LDA model with {n_topics} topics.")
    # build and fit LDA model 
    # random_state=42:reproducible results
    # learning_method='batch': full batch EM
    lda_candidate = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method="batch"
    )
    try:
        lda_candidate.fit(document_term_matrix)
    except Exception as training_error:
        print(f"  ERROR: LDA training failed for {n_topics} topics — {training_error}")
        continue
    #store model and compute perplexity
    trained_lda_models[n_topics] = lda_candidate
    model_perplexity[n_topics]   = lda_candidate.perplexity(document_term_matrix)
    print(f"Perplexity ({n_topics} topics): {model_perplexity[n_topics]:.2f}")


Training LDA model with 8 topics.
Perplexity (8 topics): 2316.05

Training LDA model with 9 topics.
Perplexity (9 topics): 2276.20

Training LDA model with 10 topics.
Perplexity (10 topics): 2240.18


In [12]:
#Topic Coherence via NPMI -(Safe for CPU running)
# Co-occurrence is computed as binary document level presence (not a sliding
# window), and the final score is mean NPMI across top word pairs per topic.
# Memory safety note:
#   The co occurrence matrix is built ONLY over the small set of unique top
#   words across all topics (~80-100 words), NOT the full 35k vocabulary.
#   Full vocabulary would produce a 35917×35917 matrix
import numpy as np
from itertools import combinations
from sklearn.feature_extraction.text import CountVectorizer as CooccurrenceVectorizer
def compute_npmi_coherence_fast(lda_model, feature_names, document_strings,
                                n_top_words=10):
    """
    Compute mean NPMI coherence across all topics using vectorized co occurrence.

    Co occurrence is measured at document level (binary presence): a word pair
    (w_i, w_j) is considered co-occurring if both appear in the same document.
    NPMI is then computed for each top word pair within each topic and averaged.
    Parameters
    ----------
    lda_model        : fitted LatentDirichletAllocation
    feature_names    : array of vocabulary strings from CountVectorizer
    document_strings : list of preprocessed document strings (one per doc)
    n_top_words      : top words per topic to evaluate (default 10)
    Returns
    -------
    float : mean NPMI coherence score across all topics, in range [-1, +1]
    """
    # collect unique top words across ALL topics
    # Only these words need co occurrence statistics not the full vocabulary.
    all_top_words    = set()
    topics_top_words = []   # list of lists, one per topic
    for topic_weight_vector in lda_model.components_:
        top_indices = topic_weight_vector.argsort()[:-n_top_words - 1:-1]
        top_words   = [feature_names[i] for i in top_indices]
        topics_top_words.append(top_words)
        all_top_words.update(top_words)
    focused_vocabulary = sorted(all_top_words)   # deterministic ordering

    # Safety check against accidentally passing the full vocabulary
    if len(focused_vocabulary) > 500:
        raise ValueError(
            f"focused_vocabulary has {len(focused_vocabulary)} words. "
            f"Expected ~80-100 (top {n_top_words} words × number of topics). "
            f"Do not pass the full CountVectorizer vocabulary to this function."
        )
    # build binary document-term matrix for top words only
    # binary=True: cell = 1 if word appears in document, 0 otherwise.
    # Shape: (n_documents, len(focused_vocabulary)) — tiny, safe in RAM.
    focused_vectorizer = CooccurrenceVectorizer(
        vocabulary = focused_vocabulary,
        binary     = True
    )
    try:
        binary_doc_term = focused_vectorizer.fit_transform(document_strings)
    except Exception as vec_error:
        print(f"Vectorization failed: {vec_error}")
        return None
    n_docs  = binary_doc_term.shape[0]
    epsilon = 1e-12
    # P(w) for each word in focused vocabulary - shape: (n_focused_words,)
    word_prob = np.asarray(binary_doc_term.sum(axis=0)).flatten() / n_docs
    # P(w_i, w_j) — document level co occurrence via sparse dot product.
    # Shape: (n_focused_words, n_focused_words) — small vocabulary
    cooccurrence_matrix = (binary_doc_term.T.dot(binary_doc_term)).toarray() / n_docs
    # Word -> column index lookup
    word_to_idx = {word: idx for idx, word in enumerate(focused_vocabulary)}
    #compute mean NPMI per topic
    topic_coherence_scores = []
    for top_words in topics_top_words:
        pair_npmi_scores = []
        for word_a, word_b in combinations(top_words, 2):
            idx_a = word_to_idx.get(word_a)
            idx_b = word_to_idx.get(word_b)
            if idx_a is None or idx_b is None:
                continue
            prob_a  = word_prob[idx_a] + epsilon
            prob_b  = word_prob[idx_b] + epsilon
            prob_ab = cooccurrence_matrix[idx_a, idx_b] + epsilon
            pmi  = np.log(prob_ab / (prob_a * prob_b))
            npmi = pmi / (-np.log(prob_ab))
            pair_npmi_scores.append(npmi)
        topic_coherence_scores.append(
            np.mean(pair_npmi_scores) if pair_npmi_scores else 0.0
        )

    return float(np.mean(topic_coherence_scores))
#run coherence evaluation for all three models 
coherence_scores = {}
for n_topics, lda_model in trained_lda_models.items():
    try:
        score = compute_npmi_coherence_fast(
            lda_model        = lda_model,
            feature_names    = vocabulary_feature_names,
            document_strings = stemmed_document_strings,
            n_top_words      = 10
        )
        coherence_scores[n_topics] = score
        print(f"NPMI Coherence ({n_topics} topics): {score:.4f}")
    except Exception as coherence_error:
        print(f"Coherence failed for {n_topics} topics: {coherence_error}")
        coherence_scores[n_topics] = None
#combined evaluation summary table 
print("\nEvaluation Summary")
print(f"{'Topics':<10} {'Perplexity':>12} {'NPMI Coherence':>16}")
for n_topics in [8, 9, 10]:
    perp    = model_perplexity[n_topics]
    coh     = coherence_scores.get(n_topics)
    coh_str = f"{coh:.4f}" if coh is not None else "N/A"
    print(f"{n_topics:<10} {perp:>12.2f} {coh_str:>16}")

NPMI Coherence (8 topics): 0.1879
NPMI Coherence (9 topics): 0.1775
NPMI Coherence (10 topics): 0.1788

Evaluation Summary
Topics       Perplexity   NPMI Coherence
8               2316.05           0.1879
9               2276.20           0.1775
10              2240.18           0.1788


## Step 10 - Quantitative Evaluation: Perplexity + Topic Coherence
### Perplexity

| Topics | Perplexity |
|--------|------------|
| 8      | 2316.05    |
| 9      | 2276.20    |
| 10     | 2240.18    |

Perplexity decreases monotonically as the number of topics increases. This is expected: more topics give the model more freedom to fit the training data, so perplexity will nearly always improve with more components. This makes perplexity an **incomplete sole metric**,a model with 50 topics would scoreeven lower, but the resulting topics would be uninterpretable.

### Topic Coherence (NPMI)
Coherence measures how semantically similar the top words within each topic areto each other. Unlike perplexity, coherence does not always improve with more topics, it peaks when topics are both internally consistent and distinct fromone another. In this run, the 8 topic model achieved the highest coherence(0.1879), while the 10 topic model scored slightly lower (0.1788). This divergence between the two metrics is a known characteristic of LDA evaluation and is discussed in the model selection step below.

> **Implementation note:** Coherence is computed using a vectorized NPMI
> (Normalised Pointwise Mutual Information) implementation based on binary
> document level co occurrence of top words per topic. This avoids importing
> gensim or scipy, which caused a DLL memory error on this machine, while
> a comparable coherence signal.

## Step 11 - Model Selection

The **10 topic model** was selected based on a combined reading of both
evaluation metrics, with an explicit acknowledgement of the tradeoff between
them.

| Topics | Perplexity | NPMI Coherence |
|--------|------------|----------------|
| 8      | 2316.05    | **0.1879**     |
| 9      | 2276.20    | 0.1775         |
| 10     | **2240.18**| 0.1788         |

1. **Perplexity** consistently improved from 8 to 10 topics (2316.05 -> 2240.18), confirming that the 10 topic model achieves the best statistical
   fit to the corpus.

2. **NPMI Coherence** peaked at 8 topics (0.1879) and was marginally lower at10 topics (0.1788), a difference of less than 0.01. This is a small and
   expected variation: adding more topics can improve statistical fit while slightly splitting some coherent clusters into more granular sub topics.

Since the coherence difference is negligible (0.009) and the 10 topic model produces visibly more fine grained and interpretable research themes, the
**10 topic model** was selected as the final model. This also illustrates an important principle in topic modeling: neither metric alone is sufficient, and the final choice requires human judgment about interpretability alongside quantitative scores.

The selected model is aliased as `best_lda_model` for clarity in downstream cells.

In [13]:
# Select the 10 topic model as the best candidate
SELECTED_N_TOPICS = 10
best_lda_model = trained_lda_models[SELECTED_N_TOPICS]

print(f"Selected model: {SELECTED_N_TOPICS} topics")
print(f"Perplexity : {model_perplexity[SELECTED_N_TOPICS]:.2f}")
print(f"NPMI Score  : {coherence_scores.get(SELECTED_N_TOPICS, 'N/A')}")

Selected model: 10 topics
Perplexity : 2240.18
NPMI Score  : 0.17881481906888877


## Step 12 - Topic Word Analysis
Two complementary views of each topic are produced:
1. **Raw frequency** (`display_top_words_raw`) — the words that appear most often within the topic component matrix, equivalent to the dominant signal
2. **Relative frequency via `LDA_Evaluator`** — words that are disproportionately concentrated in a topic compared to their corpus wide average; this surfaces the words that truly *distinguish* one topic from another and is particularly useful for resolving cases where the same word (e.g., `cell`) ranks highly across multiple topics

In [14]:
def display_top_words_raw(lda_model, feature_names, n_top_words=10):
    """
    Print the top `n_top_words` highest weight words for each topic
    in the given LDA model using raw component weights (argsort).
    Parameters
    ----------
    lda_model     : fitted LatentDirichletAllocation
    feature_names : array of vocabulary strings from CountVectorizer
    n_top_words   : how many words to show per topic (default 10)
    """
    for topic_idx, topic_weight_vector in enumerate(lda_model.components_):
        # argsort()[:-n-1:-1] returns indices of the n highest values in descending order
        top_word_indices = topic_weight_vector.argsort()[:-n_top_words - 1:-1]
        top_word_labels  = [feature_names[i] for i in top_word_indices]
        print(f"Topic {topic_idx + 1}:")
        print(", ".join(top_word_labels))
        print()
# Display raw top words for the selected 10 topic model
display_top_words_raw(best_lda_model, vocabulary_feature_names)

Topic 1:
diseas, caus, cell, lesion, occur, blood, includ, tissu, increas, clinic

Topic 2:
patient, studi, infect, respiratori, case, hospit, influenza, clinic, sever, diseas

Topic 3:
rna, protein, viral, cell, replic, viru, gene, infect, sequenc, genom

Topic 4:
model, case, time, health, diseas, studi, number, system, inform, effect

Topic 5:
sequenc, viru, cov, gene, virus, genom, strain, host, human, protein

Topic 6:
cell, infect, express, mice, immun, respons, activ, induc, viru, cd

Topic 7:
protein, cell, activ, bind, structur, membran, peptid, interact, acid, residu

Topic 8:
sampl, detect, viru, infect, pcr, test, assay, antibodi, ml, cultur

Topic 9:
health, countri, public, nation, china, global, govern, viru, diseas, infect

Topic 10:
infect, diseas, anim, studi, human, vaccin, calv, group, day, caus



In [15]:
# Use LDA_Evaluator from helper_functions to surface RELATIVE frequency top words
# Relative frequency = how much more a word appears in this topic vs. corpus average

lda_evaluator = LDA_Evaluator(
    lda_model=best_lda_model,
    vectorizer=corpus_vectorizer
)
print("Relative Frequency Top Words (LDA_Evaluator)")
print("These words are most DISTINCTIVE to each topic relative to the full corpus.\n")
for topic_col_idx in range(SELECTED_N_TOPICS):
    print(f"Topic {topic_col_idx + 1} (relative frequency)")
    try:
        rel_freq_df = lda_evaluator.eval_rel_frequency(
            topic=topic_col_idx,
            num_words=10,
            threshold=0.5    # only consider words in the top 50% by corpus frequency rank
        )
        print(list(rel_freq_df.index.values))
    except Exception as eval_error:
        print(f"  Could not compute relative frequency: {eval_error}")
    print()

Relative Frequency Top Words (LDA_Evaluator)
These words are most DISTINCTIVE to each topic relative to the full corpus.

Topic 1 (relative frequency)
['pacient', 'caso', 'infecci', 'tratamiento', 'pued', 'estudio', 'colic', 'stico', 'tico', 'nico']

Topic 2 (relative frequency)
['icu', 'lrti', 'ecmo', 'urti', 'ist', 'werden', 'oder', 'ili', 'hsct', 'sind']

Topic 3 (relative frequency)
['prf', 'bzip', 'ppmo', 'pseudoknot', 'biorxiv', 'frameshift', 'anticodon', 'caastv', 'shutoff', 'ddx']

Topic 4 (relative frequency)
['airport', 'journalist', 'airlin', 'cfd', 'curriculum', 'audienc', 'seir', 'delphi', 'semant', 'aviat']

Topic 5 (relative frequency)
['ptov', 'isav', 'contig', 'tcov', 'otu', 'diub', 'batcov', 'btov', 'sarsr', 'qx']

Topic 6 (relative frequency)
['cxcl', 'treg', 'tlr', 'ifn', 'splenocyt', 'ikk', 'traf', 'microglia', 'erk', 'ptx']

Topic 7 (relative frequency)
['aptam', 'hrc', 'mtase', 'nmr', 'lycorin', 'tgn', 'oxo', 'ergic', 'casd', 'agnp']

Topic 8 (relative frequency)

## How I Arrived at These Conclusions
### Model Selection
I trained three LDA models with 8, 9, and 10 topics and evaluated each using
two metrics:

- **Perplexity** measures how well the model fits the data statistically. Lower is better. My results were 2316.05 (8 topics), 2276.20 (9 topics),
  and 2240.18 (10 topics), so 10 topics had the best statistical fit.

- **NPMI Coherence** measures how semantically meaningful the words within each topic are relative to each other. Higher is better. My results were
  0.1879 (8 topics), 0.1775 (9 topics), and 0.1788 (10 topics). Coherence was highest for 8 topics in this run, not 10 topics. This means the two
  metrics pointed in different directions, perplexity favored 10 topics while coherence favored 8 topics.

This kind of disagreement is normal in LDA evaluation and is actually a useful finding: perplexity rewards statistical complexity while coherence measures human interpretable word groupings. Since the coherence difference between 8 and 10 topics was less than 0.01, essentially within noise,and the 10 topic model produced more specific and distinguishable research themes, I selected 10 topics as the final model.

### Topic Interpretation

To interpret each topic I used two outputs from the model:

1. **Raw top words** - the 10 highest weighted words in each topic directly
   from the LDA component matrix. These show what words dominate each topic.

2. **Relative frequency words** (via `LDA_Evaluator`) - words that appear
   disproportionately more in one topic compared to the rest of the corpus.
   These helped identify what makes each topic *unique*, especially when the
   same word like `cell`, `infect`, or `protein` appeared across multiple
   topics in the raw output.

By reading both lists together, I could form a clearer picture of each topic. For example, Topics 3 and 5 both involve `sequenc` and `gene` in their raw words, but the relative frequency output for Topic 3 shows RNA frameshift and pseudoknot terms, while Topic 5 shows named virus strain identifiers which led me to label them as molecular replication mechanisms versus viral genomics and strain surveillance respectively.

For Topic 1, the relative frequency output is dominated by Spanish words like `pacient`, `caso`, and `tratamiento`, revealing that the LDA model grouped Spanish language clinical publications together with English disease research.This shaped my interpretation of that topic as partially a multilingual data artifact. A language detection and filtering step before vectorization would be a practical improvement for future iterations of this analysis.

Bigrams, trigrams, and word clouds were considered but not included in this analysis, as the unigram vocabulary produced sufficiently interpretable topics without the additional complexity.

# Topic Interpretation - Selected 10 Topic LDA Model
> **Note on stemmed forms:** All top words are Porter stemmed tokens produced
> by the preprocessing pipeline. The stemmer strips word endings so that
> *infection*, *infected*, and *infecting* all become `infect`. Readable
> equivalents are shown in parentheses where the stemmed form is unclear.
## Topic 1 - Spanish Language Clinical Documents
**Most Common Word:** `diseas` (*disease*)
**Full Top 10:** `diseas, caus, cell, lesion, occur, blood, includ, tissu, increas, clinic`
**Relative Frequency:** `pacient, caso, infecci, tratamiento, pued, estudio, colic, stico, tico, nico`

Looking at the raw top words alone, this appears to be about disease and tissue damage in clinical settings. But the relative frequency output tells a different story, every word in that list is Spanish: `pacient` means *patient*, `caso`means *case*, `tratamiento` means *treatment*, and `estudio` means *study*.This means the LDA model did not find a clean scientific theme here instead it grouped together documents written in Spanish, because those documents shared vocabulary patterns that were distinct from the English language majority of the corpus. From a data perspective, this is a classic example of what happens when language is not controlled for before topic modeling. What makes this topic stand out is that the grouping signal is linguistic, not scientific.
## Topic 2 - Hospital Patient Outcomes and Severity Tracking
**Most Common Word:** `patient` (*patient*)
**Full Top 10:** `patient, studi, infect, respiratori, case, hospit, influenza, clinic, sever, diseas`
**Relative Frequency:** `icu, lrti, ecmo, urti, ist, werden, oder, ili, hsct, sind`

This topic clusters documents that track what happens to patients in hospital settings during respiratory illness outbreaks. The raw words `patient`,
`hospit`, `case`, `sever`, and `clinic` read like a structured dataset of patient records severity levels, case counts, clinical observations. The
relative frequency adds detail: `icu` is Intensive Care Unit, `lrti` and `urti` are lower and upper respiratory tract infection classifications, and `ecmo` is a last-resort life support intervention. There are also German words present (`werden`, `oder`, `sind`), another multilingual artifact similar to Topic 1, but less dominant.
## Topic 3 - Virus Replication Machinery at the Molecular Level
**Most Common Word:** `rna` (*RNA*)
**Full Top 10:** `rna, protein, viral, cell, replic, viru, gene, infect, sequenc, genom`
**Relative Frequency:** `prf, bzip, ppmo, pseudoknot, biorxiv, frameshift, anticodon, caastv, shutoff, ddx`
The raw words here `rna`, `replic`, `gene`, `genom`, `sequenc` are all about the internal instructions a virus uses to copy itself inside a cell. Think
of it like reverse engineering the source code of a virus: RNA is the instruction set, replication is the execution, and the genome is the full
codebase. The relative frequency terms go even deeper into that analogy `frameshift` and `pseudoknot` are structural tricks viruses use to make their
instruction set more compact, similar to how compressed data formats work. What makes this topic distinct from Topic 5 which also has sequencing terms
is that Topic 3 is about the *execution process* of how the virus runs, while Topic 5 is about *classifying and cataloguing* different virus versions.
## Topic 4 - Mathematical Modeling and Outbreak Forecasting
**Most Common Word:** `model` (*model*)
**Full Top 10:** `model, case, time, health, diseas, studi, number, system, inform, effect`
**Relative Frequency:** `airport, journalist, airlin, cfd, curriculum, audienc, seir, delphi, semant, aviat`
The vocabulary — `model`, `number`, `time`, `system`, `effect` describes simulation and forecasting work. The relative frequency
output adds context: `seir` is a standard compartmental epidemic model (Susceptible, Exposed, Infected, Recovered), equivalent to a state machine
with transition probabilities. `delphi` refers to a structured expert forecasting method. `airport` and `airlin` indicate studies modeling how
diseases spread through transportation networks essentially graph based transmission modeling. This topic stands out because it is the only one where
the primary output is a prediction or simulation rather than a lab measurement or clinical observation.
## Topic 5 - Virus Strain Classification and Genome Surveillance
**Most Common Word:** `sequenc` (*sequencing*)
**Full Top 10:** `sequenc, viru, cov, gene, virus, genom, strain, host, human, protein`
**Relative Frequency:** `ptov, isav, contig, tcov, otu, diub, batcov, btov, sarsr, qx`
This topic is about cataloguing and classifying different virus strains by reading and comparing their genomes the bioinformatics equivalent of version
control and diff ing across a large codebase. Words like `strain`, `host`,`cov` (coronavirus), and `sequenc` describe a surveillance workflow: collect
samples, read the genome, compare against known strains, classify. The relative frequency output is a list of named virus strain identifiers batcov` is BatCoronavirus, `tcov` is Turkey Coronavirus, `sarsr` is SARS-related like version tags in a repository. This topic stands out because it is the broadest cross species virus tracking topic in the model, and represents exactly the kind of database work that made it possible to identify a new coronavirus quickly in early 2020.
## Topic 6 - Immune System Activation in Lab Experiments
**Most Common Word:** `cell` (*cell*)
**Full Top 10:** `cell, infect, express, mice, immun, respons, activ, induc, viru, cd`
**Relative Frequency:** `cxcl, treg, tlr, ifn, splenocyt, ikk, traf, microglia, erk, ptx`
This topic clusters controlled laboratory experiments that measure how the immune system reacts when cells are exposed to a virus. The key signals here
are `mice` (experiments done in mouse models rather than humans), `activ`(*activation*), `induc` (*induction*), and `respons` (*response*) — these together describe a cause and effect experimental pattern: expose cells to virus, measure what gets switched on or off. What makes this topic distinct from Topic 2 is the scale.Topic 2 is about human patients in hospitals, Topic 6 is about individual cells in a lab dish or a mouse, zoomed in to the mechanism level.
## Topic 7 - Protein Shape and Molecule Interaction Studies
**Most Common Word:** `protein` (*protein*)
**Full Top 10:** `protein, cell, activ, bind, structur, membran, peptid, interact, acid, residu`
**Relative Frequency:** `aptam, hrc, mtase, nmr, lycorin, tgn, oxo, ergic, casd, agnp`
This topic is about the physical shape of viral proteins and how they fit together with other molecules essentially 3D structure matching. Words like
`bind`, `structur`, `interact`, and `residu` describe research that asks: what does this protein look like, and what does it attach to? . This is directly relevant to drug design because knowing the shape of a virus protein tells you what kind of molecule could block it. What makes
this topic distinct from Topic 3 is that Topic 3 is about how the virus *runs its code*, while Topic 7 is about the physical *hardware components*
that code produces.
## Topic 8 - Lab Detection and Diagnostic Testing Procedures
**Most Common Word:** `sampl` (*sample*)
**Full Top 10:** `sampl, detect, viru, infect, pcr, test, assay, antibodi, ml, cultur`
**Relative Frequency:** `igi, rcv, photocatalyt, photocatalysi, nasba, photocatalyst, decaro, parasui, crcov, mgb`
This topic groups together documents describing how to detect whether a virus is present in a collected sample. The vocabulary `sampl`, `detect`, `pcr`,`test`, `assay`, `cultur' reads like a testing pipeline: collect a sample, run a test protocol, measure the result. `pcr` is the standard virus detection technique (Polymerase Chain Reaction) that became widely known during the COVID 19 pandemic for diagnostic testing. This topic stands out because it is the most procedural and methods oriented cluster in the model.It is about *the detection workflow itself* rather than what the virus does or how patients are affected. 
## Topic 9 - International Policy and Cross-Country Outbreak Response
**Most Common Word:** `health` (*health*)
**Full Top 10:** `health, countri, public, nation, china, global, govern, viru, diseas, infect`
**Relative Frequency:** `chez, cett, enfant, tude, peut, autr, traitement, risqu, cliniqu, deux`
The raw top words `countri`, `nation`, `global`, `govern`, `china`, `public`point to international and governmental responses to disease outbreaks. However the relative frequency output is entirely French: `chez` means *at/in*, `enfant` means *child*, `traitement` means *treatment*, and `cliniqu` means *clinical*. This means Topic 9, like Topic 1 (Spanish) and 
Topic 2 (German), has partially absorbed a cluster of French-language publications. The raw  signal is genuine there is real international policy content here but the most distinctive vocabulary in this topic comes from French documents. From a data perspective this is the third multilingual artifact in the model, reinforcing that language detection before vectorization 
would significantly improve topic purity across this corpus. Unlike Topic 4 which is about mathematical modeling of spread, Topic 9 is concerned with the governance and communication layer of outbreak response, and unlike Topics 1 and 2, the multilingual signal here sits on top of a genuine policy focused English language cluster rather than replacing it entirely.
## Topic 10 - Animal to Human Disease Transmission Research
**Most Common Word:** `infect` (*infection*)
**Full Top 10:** `infect, diseas, anim, studi, human, vaccin, viru, immun, transm, outbreak`
**Relative Frequency:** `heifer, mhf, scour, vcjd, bushmeat, bioga, feedlot, cordycep, cockroach, colostr`
This topic covers research on how diseases move from animals to humans zoonotic transmission. The raw words `anim`, `human`, `transm`, and `outbreak`
describe the cross species spread pattern at a general level. The relative frequency output makes the topic more specific: `heifer`, `feedlot`, and
`scour` point to cattle and livestock farming contexts, `bushmeat` and `cockroach` point to wildlife and environmental exposure pathways, and `vcjd`
(variant Creutzfeldt Jakob Disease, the human form of mad cow disease) confirms this topic includes agricultural disease crossover research. From a data perspective, the relative frequency words clarify that this is not just broadly about any animal-to-human transmission it is specifically anchored in livestock, agricultural settings, and wildlife contact. What makes this topic stand out is that it sits at the boundary between the veterinary topics and the human clinical topics, studying the transfer event between species, which is the critical edge in any zoonotic outbreak including COVID 19.
# Overall Topic Summary
The 10 topic LDA model reveals three natural groupings across the pre pandemic coronavirus research corpus. Four topics operate at the molecular and lab level.
— Topic 3 covers how viruses replicate, Topic 5 catalogues virus strains,
Topic 6 measures immune responses in lab settings, and Topic 7 maps protein
structures for drug design. Four topics cover the clinical and population level
— Topic 2 tracks patient outcomes in hospitals, Topic 8 describes detection
testing pipelines, Topic 9 documents international policy responses, and
Topic 10 studies animal to human transmission. The remaining two topics reveal
data quality characteristics of the corpus itself: Topic 1 shows that
Spanish language publications were grouped by language rather than theme, and
Topic 4 shows that mathematical forecasting models form their own distinct
cluster. As a whole, the model shows that the scientific community had built a
multi layered research foundation before COVID 19 emerged, from molecular
level virus replication studies all the way up to international governance
frameworks and that this corpus, while rich, would benefit from language
filtering as a preprocessing step before topic modeling.